# 🖥️ CROCO tutorial

This tutorial will show how to run a 3D simulation with output from the CROCO model.

## Example setup

We start with loading the relevant modules and the data. We use the [`xroms`](https://xroms.readthedocs.io/) package to add the vertical coordinates (depth as a function of time and space) to the dataset.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import xroms

import parcels
import parcels.tutorial

ds_fields = parcels.tutorial.open_dataset("CROCOidealized_data/data")

In [ ]:
# Add ROMS/CROCO vertical coordinates and grid metrics
ds_fields, xgrid = xroms.roms_dataset(ds_fields)
ds_fields.xroms.set_grid(xgrid)

The simulation will be a very simple, idealised flow: a purely zonal flow over a sloping bottom. This flow (which is somewhat unrealistic of course) nicely showcases that particles stay on their initial depth levels, even though the sigma-layers slope down. 

This flow has been created by first running the example from the [Shelf front example on the CROCO website](https://croco-ocean.gitlabpages.inria.fr/croco_doc/model/model.test_cases.shelfront.html). Then, we took the restart file are manually replaced all the `u`-velocities with `1` m/s and all the `v`-velocities with `0` m/s. This way we get a purely zonal flow. We then started a new simulation from the restart file, and CROCO then automatically calculated the `w` velocities to match the new zonal field. We saved the `time=0` snapshot from this new run and use it below.

Now we create a FieldSet object using the {py:func}`parcels.convert.croco_to_sgrid()` function to first create an S-Grid compliant datatset, and then use that in  {py:func}`parcels.FieldSet.from_sgrid_conventions()` to create the FieldSet.

In [ ]:
fields = {
    "U": ds_fields["u"],
    "V": ds_fields["v"],
    "W": ds_fields["w"],
    "depth": ds_fields["z_w_psi"],
}

ds_fset = parcels.convert.croco_to_sgrid(fields=fields, coords=ds_fields)
ds_fset.time.attrs["axis"] = "T"

fieldset = parcels.FieldSet.from_sgrid_conventions(ds_fset)

# Convert the FieldSet to windowed arrays for better performance
fieldset = fieldset.to_windowed_arrays()

Now we can use this Fieldset to advect particles as we would normally do. Note that the particle depths should be provided in (negative) meters, not in sigma-coordinates.

In [ ]:
X, Z = np.meshgrid(
    [40e3, 80e3, 120e3],
    [100, -10, -130, -250, -400, -850, -1400, -1550],
)
Y = np.ones(X.size) * 98000


def DeleteParticle(particles, fieldset):
    any_error = particles.state >= 50
    particles[any_error].state = parcels.StatusCode.Delete


pset = parcels.ParticleSet(fieldset=fieldset, pclass=parcels.Particle, x=X, y=Y, z=Z)

outputfile = parcels.ParticleFile(
    path="croco_particles3D.parquet",
    outputdt=np.timedelta64(5000, "s"),
    mode="w",
)

pset.execute(
    [parcels.kernels.AdvectionRK2_3D, DeleteParticle],
    runtime=np.timedelta64(50_000, "s"),
    dt=np.timedelta64(100, "s"),
    output_file=outputfile,
)

Now we plot the particle trajectories below. Note that the particles follow the sloping sigma-layers. Also note that particles released above the surface (where depth >0) or below the bathymetry are not advected (due to the `DeleteParticle` kernel).

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
df = pl.read_parquet("croco_particles3D.parquet")

ax.plot(X / 1e3, Z, "k.", label="Initial positions")
for traj in df.partition_by("particle_id", maintain_order=True):
    ax.plot(traj["x"] / 1e3, traj["z"], ".-")

ax.plot(
    fieldset.depth.grid.lon[0, :-1] / 1e3,
    fieldset.depth.data[0, :, 25, :].T,
    "k",
    linewidth=0.5,
)
ax.set_xlabel("X [km]")
ax.set_xlim(30, 170)
ax.set_ylabel("Depth [m]")
ax.set_title("Particles in idealized CROCO velocity field using AdvectionRK2_3D_CROCO")
plt.tight_layout()
plt.show()

### A CROCO simulation with no vertical velocities

It may be insightful to compare this 3D run with the `AdvectionRK2_3D_CROCO` kernel with a run where the vertical velocity (`W`) is set to zero. In that case, the particles will stay on their initial depth levels and _not_ follow sigma-layers.

In [ ]:
fieldset_noW = parcels.FieldSet.from_sgrid_conventions(ds_fset)
fieldset_noW.W.data[:] = 0.0
fieldset_noW = fieldset_noW.to_windowed_arrays()

X, Z = np.meshgrid(
    [40e3, 80e3, 120e3],
    [100, -10, -130, -250, -400, -850, -1400, -1550],
)
Y = np.ones(X.size) * 100

pset_noW = parcels.ParticleSet(
    fieldset=fieldset_noW, pclass=parcels.Particle, x=X, y=Y, z=Z
)

outputfile = parcels.ParticleFile(
    path="croco_particles_noW.parquet", outputdt=np.timedelta64(5000, "s"), mode="w"
)

pset_noW.execute(
    [parcels.kernels.AdvectionRK2_3D, DeleteParticle],
    runtime=np.timedelta64(50_000, "s"),
    dt=np.timedelta64(100, "s"),
    output_file=outputfile,
)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
df = pl.read_parquet("croco_particles_noW.parquet")

ax.plot(X / 1e3, Z, "k.", label="Initial positions")
for traj in df.partition_by("particle_id", maintain_order=True):
    ax.plot(traj["x"] / 1e3, traj["z"], ".-")

ax.plot(
    fieldset.depth.grid.lon[0, :-1] / 1e3,
    fieldset.depth.data[0, :, 25, :].T,
    "k",
    linewidth=0.5,
)
ax.set_xlabel("X [km]")
ax.set_xlim(30, 170)
ax.set_ylabel("Depth [m]")
ax.set_title("Particles in idealized CROCO velocity field with W=0")
plt.tight_layout()
plt.show()